# Ingesta y Capa Bronce

En esta notebook se inicia la construcción del pipeline de datos, trabajando con el archivo de datos crudos provistos por el SGC-OVSPA.
El objetivo es cargar el dataset completo y particionarlo por estación en una estructura de carpetas (Capa Bronce), preservando los datos crudos.

## Importar las librerías necesarias

In [1]:
import pandas as pd
import os
from pathlib import Path

print("Importación de librerías completada.")

Importación de librerías completada.


## Configuración de paths y carpetas del proyecto

In [9]:
BASE_DIR = Path('..').resolve()
RAW_DIR = BASE_DIR / 'data' / 'raw'
BRONCE_DIR = BASE_DIR / 'data' / 'bronce'

# Crear carpetas si no existen
for path in [RAW_DIR, BRONCE_DIR]:
    path.mkdir(parents=True, exist_ok=True)
    
print("Iniciación de carpetas del proyecto completada.")

#/Users/andavaro/Documents/Andrés/MCB/GVD/epever_soh.csv

Iniciación de carpetas del proyecto completada.


## Lectura del archivo de datos crudos

Cargaremos el archivo CSV original que contiene los datos combinados de múltiples estaciones.

In [13]:
# Ruta del archivo
archivo_datos = RAW_DIR/'epever_soh.csv'

# Leer el archivo CSV
df_crudo = pd.read_csv(archivo_datos)
print(f"Dataset cargado correctamente. Total de registros: {len(df_crudo)}")


Dataset cargado correctamente. Total de registros: 106872


## Exploración inicial y obtención de estaciones

In [14]:
# Obtener la lista de estaciones únicas
df_crudo['station_name'] = df_crudo['station_name'].replace({'SJose': 'SanJose'})
estaciones_unicas = df_crudo['station_name'].dropna().unique()

print(f"Cantidad de estaciones encontradas: {len(estaciones_unicas)}")
print("Estaciones disponibles:", estaciones_unicas)

# Mostrar las primeras filas y los tipos de datos
print("\nPrimeras 5 filas del dataset:")
display(df_crudo.head())

print("\nTipos de datos de las columnas:")
print(df_crudo.dtypes)


Cantidad de estaciones encontradas: 22
Estaciones disponibles: <StringArray>
[      'Crater',         'Vivi',      'Mallama',         'Coba',
      'Tosoabi',      'SanJose',        'Bruma',   'PajaBlanca',
 'CruzAmarillo',  'Bordoncillo',     'Barranco',  'AguasAgrias',
      'Cufinho',        'Arles',      'Limones',   'MundoNuevo',
       'LaMesa',      'Paisita',  'SanCayetano', 'MijitayoBajo',
      'Anganoy',      'Consaca']
Length: 22, dtype: str

Primeras 5 filas del dataset:


,station_name,timestamp,pv_voltage_V,pv_current_A,pv_power_W,battery_voltage_V,battery_current_A,battery_soc_%,controller_temp_C,charging_status,load_current_A,load_power_W,energy_generated_today_kWh,energy_consumed_today_kWh,battery_voltage_max_today_V,battery_voltage_min_today_V,energy_total_generated_kWh,load_voltage_V
0,Crater,2026-03-12 08:42:47,106.03,1.47,15551.0,14.36,10.84,91.0,25.0,11.0,1.75,2656.0,0.62,0.61,14.5,12.56,208.28,NaN
1,Vivi,2026-03-12 08:42:47,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Crater,2026-03-12 08:43:59,106.18,1.47,15760.0,14.38,10.93,91.0,25.0,11.0,1.82,2559.0,0.62,0.61,14.5,12.56,208.28,NaN
3,Vivi,2026-03-12 08:43:59,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Vivi,2026-03-12 08:44:36,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Tipos de datos de las columnas:
station_name                       str
timestamp                          str
pv_voltage_V                   float64
pv_current_A                   float64
pv_power_W                     float64
battery_voltage_V              float64
battery_current_A              float64
battery_soc_%                  float64
controller_temp_C              float64
charging_status                float64
load_current_A                 float64
load_power_W                   float64
energy_generated_today_kWh     float64
energy_consumed_today_kWh      float64
battery_voltage_max_today_V    float64
battery_voltage_min_today_V    float64
energy_total_generated_kWh     float64
load_voltage_V                 float64
dtype: object


## Procesamiento: Partición de datos por estación para la Capa Bronce

Tenemos un solo archivo CSV con todos los registros. El objetivo de la capa bronce es mantener los datos crudos pero organizados de una forma más eficiente. Vamos a separar el dataset original y a guardar un archivo CSV por cada estación dentro de su respectiva carpeta en `data/bronce/`.

In [15]:
if not df_crudo.empty:
    errores = 0
    registros_procesados = 0

    for estacion in estaciones_unicas:
        try:
            # Limpiar el nombre de la estación para usarlo como nombre de carpeta
            nombre_clean = str(estacion).strip().lower().replace(" ", "_")
            
            # Filtrar los datos para la estación actual
            df_estacion = df_crudo[df_crudo['station_name'] == estacion]
            
            if not df_estacion.empty:
                # Crear carpeta para la estación en la Capa Bronce
                path_estacion = BRONCE_DIR / nombre_clean
                path_estacion.mkdir(parents=True, exist_ok=True)
                
                # Guardar el archivo CSV
                # Nota: Podríamos particionar por fecha también, pero al ser un dataset 
                # único, guardaremos un archivo consolidado por estación para la capa bronce.
                archivo_csv = path_estacion / f"{nombre_clean}_bronce.csv"
                df_estacion.to_csv(archivo_csv, index=False)
                
                registros = len(df_estacion)
                registros_procesados += registros
                print(f"Guardados {registros} registros para la estación '{estacion}' en {archivo_csv}")
                
        except Exception as e:
            print(f"Error al procesar la estación {estacion}: {e}")
            errores += 1

    # Reporte final
    print("\nProceso de ingesta a Capa Bronce completado.")
    print(f"Estaciones procesadas: {len(estaciones_unicas)}")
    print(f"Total de registros distribuidos: {registros_procesados}")
    if errores > 0:
        print(f"Errores encontrados: {errores}")


Guardados 9576 registros para la estación 'Crater' en /Users/andavaro/Documents/Andrés/MCB/GVD/data/bronce/crater/crater_bronce.csv
Guardados 8832 registros para la estación 'Vivi' en /Users/andavaro/Documents/Andrés/MCB/GVD/data/bronce/vivi/vivi_bronce.csv
Guardados 1818 registros para la estación 'Mallama' en /Users/andavaro/Documents/Andrés/MCB/GVD/data/bronce/mallama/mallama_bronce.csv
Guardados 9505 registros para la estación 'Coba' en /Users/andavaro/Documents/Andrés/MCB/GVD/data/bronce/coba/coba_bronce.csv
Guardados 9318 registros para la estación 'Tosoabi' en /Users/andavaro/Documents/Andrés/MCB/GVD/data/bronce/tosoabi/tosoabi_bronce.csv
Guardados 9349 registros para la estación 'SanJose' en /Users/andavaro/Documents/Andrés/MCB/GVD/data/bronce/sanjose/sanjose_bronce.csv
Guardados 8287 registros para la estación 'Bruma' en /Users/andavaro/Documents/Andrés/MCB/GVD/data/bronce/bruma/bruma_bronce.csv
Guardados 5 registros para la estación 'PajaBlanca' en /Users/andavaro/Docu

# Conclusión

En este notebook realizamos el proceso de **ingesta de datos de SoH en sistemas fotovoltáicos de estaciones del SGC-OVSPA** y la creación de la **Capa Bronce** de nuestro proyecto:

1. **Lectura del dataset consolidado** que contiene los registros de los controladores de carga Epever instalados en las estaciones.
2. **Identificación de las estaciones** presentes en los datos.
3. **Partición de los datos crudos** y organización en la estructura de carpetas del proyecto, guardando la información agrupada por estación.

Esta etapa preserva la información original sin alteraciones, sirviendo como la fuente de datos (Capa Bronce) para las futuras etapas en la Capa Plata.